# Vector-Quantized VAE (VQ-VAE) — a *discrete* latent space

> Tutorial pair for [`vq_vae.py`](vq_vae.py).

## 1. Intuition
A plain VAE has a *continuous* Gaussian latent. VQ-VAE instead keeps a small
**codebook** of $K$ learnable vectors and snaps every encoder output to its
nearest codebook entry. The latent becomes a grid of **integers** (which code
won), which is perfect for later modelling with a discrete autoregressive prior
(PixelCNN, transformers). The catch: "pick the nearest code" is an $\arg\min$ —
it has no gradient — so we need a trick to train through it.

## 2. Concept (the slide)
- **Encoder** $z_e=E(x)\in\mathbb R^D$.
- **Codebook** $\{e_1,\dots,e_K\}\subset\mathbb R^D$.
- **Quantize:** $k=\arg\min_j\|z_e-e_j\|$, output $z_q=e_k$.
- **Decoder** reconstructs $\hat x=D(z_q)$.
- **Straight-through estimator (STE):** in the backward pass pretend the
  quantizer is the identity, $\partial z_q/\partial z_e:=1$, so the decoder's
  gradient is copied straight onto the encoder.
- Two extra losses pull the codebook and the encoder together; an optional
  **EMA** update replaces the codebook loss with a running average.

## 3. Math derivation — the VQ objective & straight-through gradient

**Quantization.** With codebook $E\in\mathbb R^{K\times D}$,
$$k=\operatorname*{arg\,min}_j\|z_e-e_j\|_2^2,\qquad z_q=e_k.$$
The squared distance is expanded the cheap way,
$\|z_e-e_j\|^2=\|z_e\|^2-2\,z_e^\top e_j+\|e_j\|^2$, so the whole $(N,K)$ table is
one matmul.

**The gradient problem.** $z_q$ is a piecewise-constant function of $z_e$, so
$\partial z_q/\partial z_e=0$ almost everywhere and the encoder would never learn.

**Straight-through estimator.** Define the forward value as $z_q$ but *route the
gradient around* the quantizer. Algebraically,
$$\boxed{\,z_q^{\text{st}}=z_e+\operatorname{sg}[\,e_k-z_e\,]\,}$$
where $\operatorname{sg}[\cdot]$ is the stop-gradient. Numerically
$z_q^{\text{st}}=e_k$, but $\partial z_q^{\text{st}}/\partial z_e=1$, so
$\partial L/\partial z_e=\partial L/\partial z_q$.

**Full objective.** Reconstruction plus two vector-quantization terms:
$$\mathcal L=\underbrace{\log p(x\mid z_q)}_{\text{recon}}
 +\underbrace{\|\operatorname{sg}[z_e]-e_k\|_2^2}_{\text{codebook loss}}
 +\;\beta\underbrace{\|z_e-\operatorname{sg}[e_k]\|_2^2}_{\text{commitment loss}}.$$
- The **codebook loss** moves the chosen code $e_k$ toward the encoder output
  (it only sees $e_k$, since $z_e$ is detached).
- The **commitment loss** moves the encoder output toward its code so the
  encoder "commits" and the embeddings do not grow unboundedly; $\beta\approx0.25$.

**EMA variant.** Instead of learning the codebook by gradient descent, treat each
code as the mean of the encoder vectors assigned to it, maintained online:
$$N_k\leftarrow\gamma N_k+(1-\gamma)n_k,\qquad
  m_k\leftarrow\gamma m_k+(1-\gamma)\!\!\sum_{i:k_i=k}\!\!z_{e,i},\qquad
  e_k=\frac{m_k}{N_k},$$
with Laplace smoothing on $N_k$ to revive dead codes. This drops the codebook
loss term and is usually more stable.

**Why no KL?** With a uniform categorical prior over $K$ codes the KL to the
posterior is the constant $\log K$, so it vanishes from the gradient — the prior
is instead *learned afterwards* by an autoregressive model over the code indices.

## 4. Model — encoder, vector quantizer (STE), decoder

In [ ]:
# ===== actual implementation from vq_vae.py =====
from __future__ import annotations

import numpy as np

SEED = 0

def _straight_through_numpy(z_e, e_k):
    """forward value e_k, but pretend d z_q/d z_e = 1 (returns value only)."""
    return z_e + (e_k - z_e)

import torch

import torch.nn as nn

import torch.nn.functional as F

def get_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

class VectorQuantizer(nn.Module):
    r"""
    Snap each input vector to its nearest codebook entry.

    Codebook  E = [e_1, ..., e_K],  e_j in R^D.  For input z_e:
        k        = argmin_j || z_e - e_j ||^2
        z_q      = e_k                                  (forward)
        d z_q/d z_e := 1                                (straight-through)

    Losses (added to the reconstruction loss):
        codebook   = || sg[z_e] - e_k ||^2     (move codes toward encoder)
        commitment = || z_e - sg[e_k] ||^2     (move encoder toward codes)
    where sg[.] is stop-gradient. With EMA the codebook term is replaced by an
    EMA update of the embeddings and is not backpropagated.
    """

    def __init__(self, num_codes: int = 32, dim: int = 16,
                 commitment: float = 0.25, ema: bool = False, decay: float = 0.99):
        super().__init__()
        self.K, self.D, self.beta = num_codes, dim, commitment
        self.ema, self.decay, self.eps = ema, decay, 1e-5
        self.embedding = nn.Embedding(num_codes, dim)
        self.embedding.weight.data.uniform_(-1.0 / num_codes, 1.0 / num_codes)
        if ema:
            # EMA accumulators are buffers (not trained by the optimizer)
            self.embedding.weight.requires_grad_(False)
            self.register_buffer("cluster_size", torch.zeros(num_codes))
            self.register_buffer("ema_w", self.embedding.weight.data.clone())

    def forward(self, z_e: torch.Tensor):
        # z_e: (N, D). Squared distances to every code: ||z||^2 - 2 z.e + ||e||^2
        e = self.embedding.weight                                  # (K, D)
        dist = (z_e.pow(2).sum(1, keepdim=True)
                - 2 * z_e @ e.t()
                + e.pow(2).sum(1))                                 # (N, K)
        idx = dist.argmin(1)                                       # (N,)
        z_q = self.embedding(idx)                                  # (N, D)

        # losses
        codebook = F.mse_loss(z_q, z_e.detach())                  # move codes -> enc
        commit = F.mse_loss(z_q.detach(), z_e)                    # move enc -> codes

        if self.ema and self.training:
            self._ema_update(z_e.detach(), idx)
            vq_loss = self.beta * commit                          # no codebook grad
        else:
            vq_loss = codebook + self.beta * commit

        # straight-through: value is z_q, gradient flows to z_e unchanged
        z_q_st = z_e + (z_q - z_e).detach()

        # perplexity = effective number of codes used (a usage diagnostic)
        probs = torch.bincount(idx, minlength=self.K).float() / idx.numel()
        perplexity = torch.exp(-(probs * (probs + 1e-10).log()).sum())
        return z_q_st, vq_loss, idx, perplexity

    @torch.no_grad()
    def _ema_update(self, z_e: torch.Tensor, idx: torch.Tensor) -> None:
        onehot = F.one_hot(idx, self.K).type(z_e.dtype)           # (N, K)
        n = onehot.sum(0)                                         # counts per code
        dw = onehot.t() @ z_e                                     # (K, D) sums
        self.cluster_size.mul_(self.decay).add_(n, alpha=1 - self.decay)
        self.ema_w.mul_(self.decay).add_(dw, alpha=1 - self.decay)
        # Laplace smoothing of the counts to avoid dead/zero codes
        total = self.cluster_size.sum()
        cs = (self.cluster_size + self.eps) / (total + self.K * self.eps) * total
        self.embedding.weight.data.copy_(self.ema_w / cs.unsqueeze(1))

class VQVAE(nn.Module):
    """A tiny MLP VQ-VAE for flattened 8x8 images (in_dim=64)."""

    def __init__(self, in_dim: int = 64, hidden: int = 128, dim: int = 16,
                 num_codes: int = 32, commitment: float = 0.25, ema: bool = False):
        super().__init__()
        self.enc = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, dim))
        self.vq = VectorQuantizer(num_codes, dim, commitment, ema)
        self.dec = nn.Sequential(
            nn.Linear(dim, hidden), nn.ReLU(),
            nn.Linear(hidden, in_dim))

    def forward(self, x: torch.Tensor):
        z_e = self.enc(x)
        z_q, vq_loss, idx, perplex = self.vq(z_e)
        xhat = torch.sigmoid(self.dec(z_q))
        return xhat, vq_loss, idx, perplex

    def loss(self, x: torch.Tensor):
        xhat, vq_loss, idx, perplex = self(x)
        recon = F.binary_cross_entropy(xhat, x, reduction="none").sum(1).mean()
        return recon + vq_loss, recon, vq_loss, perplex

    def fit(self, X, epochs: int = 60, batch: int = 128, lr: float = 2e-3):
        dev = get_device()
        self.to(dev)
        X = torch.as_tensor(X, dtype=torch.float32, device=dev)
        opt = torch.optim.Adam(self.parameters(), lr=lr)
        self.history = []
        for _ in range(epochs):
            perm = torch.randperm(len(X), device=dev)
            tot = 0.0
            for s in range(0, len(X), batch):
                xb = X[perm[s:s + batch]]
                loss, recon, vq, perplex = self.loss(xb)
                opt.zero_grad(); loss.backward(); opt.step()
                tot += loss.item()
            self.history.append(tot / max(1, len(X) // batch))
        return self

    @torch.no_grad()
    def reconstruct(self, X):
        dev = next(self.parameters()).device
        X = torch.as_tensor(X, dtype=torch.float32, device=dev)
        xhat, _, idx, _ = self(X)
        return xhat.cpu().numpy(), idx.cpu().numpy()

def demo():
    np.random.seed(SEED); torch.manual_seed(SEED)
    torch.set_num_threads(1)  # tiny model: 1 thread avoids CPU thrashing
    from sklearn.datasets import load_digits
    X = (load_digits().data / 16.0).astype(np.float32)            # (1797, 64) in [0,1]

    m = VQVAE(64, num_codes=32, dim=16, ema=False).fit(X, epochs=40)
    xhat, idx = m.reconstruct(X[:512])
    mse = float(np.mean((xhat - X[:512]) ** 2))
    used = len(np.unique(idx))
    print(f"VQ-VAE (loss codebook) final loss={m.history[-1]:.3f}  "
          f"recon MSE={mse:.4f}  codes used={used}/32")

    me = VQVAE(64, num_codes=32, dim=16, ema=True).fit(X, epochs=40)
    xhat_e, idx_e = me.reconstruct(X[:512])
    mse_e = float(np.mean((xhat_e - X[:512]) ** 2))
    print(f"VQ-VAE (EMA codebook)  final loss={me.history[-1]:.3f}  "
          f"recon MSE={mse_e:.4f}  codes used={len(np.unique(idx_e))}/32")

## 5. Training / sampling — reconstruct & inspect codebook usage

In [ ]:
# ===== actual implementation from vq_vae.py =====

## 6. Train & sample on 8×8 digits (loss-codebook vs EMA)

In [ ]:
demo()

## 7. Visualization — reconstructions and codebook-usage histogram

In [ ]:
import matplotlib; matplotlib.use("Agg")
import numpy as np, matplotlib.pyplot as plt
from sklearn.datasets import load_digits
import vq_vae as M

X = (load_digits().data / 16.0).astype("float32")
m = M.VQVAE(64, num_codes=32, dim=16).fit(X, epochs=40)
xhat, idx = m.reconstruct(X[:512])

fig, axes = plt.subplots(2, 8, figsize=(12, 3))
for i in range(8):
    axes[0, i].imshow(X[i].reshape(8, 8), cmap="gray")
    axes[1, i].imshow(xhat[i].reshape(8, 8), cmap="gray")
for a in axes.ravel():
    a.axis("off")
axes[0, 0].set_title("real", loc="left"); axes[1, 0].set_title("recon", loc="left")
plt.tight_layout(); plt.show()

plt.figure(figsize=(7, 3))
plt.hist(idx, bins=np.arange(33) - 0.5, rwidth=0.9)
plt.xlabel("codebook index"); plt.ylabel("count")
plt.title(f"Codebook usage ({len(np.unique(idx))}/32 codes active)")
plt.tight_layout(); plt.show()

## 8. Takeaways & pitfalls
- The **straight-through estimator** is the whole game: it lets gradients skip a
  non-differentiable $\arg\min$ by defining the backward pass as the identity.
- **Codebook collapse**: only a few codes get used. Fixes — EMA updates with
  Laplace smoothing, reinitializing dead codes, smaller codebook, or normalizing
  embeddings.
- The commitment weight $\beta$ trades encoder "commitment" against flexibility;
  $0.25$ is the standard default.
- VQ-VAE itself is *not* a generator — it learns a discrete code. To *sample* you
  train an autoregressive prior (PixelCNN / transformer) over the code indices,
  which connects directly to the autoregressive models in this repo.